# Text Generation using RNN, LSTM and GRU

**Aim:** To build character-level language models with a Simple RNN, an LSTM and a GRU, to generate text, and to
compare the three architectures.

**Dataset:** Shakespeare text (about 1.1 million characters).

**Steps:** clean the text → map characters to numbers → create training sequences → train RNN, LSTM and GRU →
generate text → **test** (held-out loss and perplexity, next-character accuracy, quality of the generated text).

**Note on the data split.** The text is split *by position*: first 90 % for training, next 5 % for validation, last
5 % for testing. The test part is never seen during training, so the test scores are honest.

In [ ]:
!pip install -q tensorflow

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import re
import urllib.request
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(42)
np.random.seed(42)
print("TensorFlow version:", tf.__version__)

## 1. Load the text

In [ ]:
# [LOAD]
url = "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt"
file_path = "shakespeare.txt"
urllib.request.urlretrieve(url, file_path)
print("Dataset downloaded successfully.")

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

print("Total characters:", len(text))
print("\nFirst 500 characters:\n")
print(text[:500])

## 2. Clean the text and build the character vocabulary

In [ ]:
text = text.lower()

# keep letters, digits, whitespace and basic punctuation
text = re.sub(r"[^a-z0-9\s.,!?;:'\"-]", "", text)
print("Characters after cleaning:", len(text))

vocab = sorted(set(text))
vocab_size = len(vocab)
print("Vocabulary size:", vocab_size)
print("Vocabulary:", vocab)

char_to_index = {char: index for index, char in enumerate(vocab)}
index_to_char = np.array(vocab)

print("\nCharacter -> Index (first 20)")
for char in vocab[:20]:
    print(repr(char), "→", char_to_index[char])

In [ ]:
text_as_int = np.array([char_to_index[char] for char in text], dtype=np.int32)
print(text_as_int[:100])

## 3. Create training, validation and test sequences

In [ ]:
sequence_length = 100     # characters per training sequence
step = 10                 # distance between the starts of two sequences (covers the WHOLE text)

n = len(text_as_int)
train_end = int(n * 0.90)
val_end = int(n * 0.95)

def make_sequences(data):
    windows = np.lib.stride_tricks.sliding_window_view(data, sequence_length)[::step]
    windows = np.array(windows)
    return windows[:, :-1], windows[:, 1:]       # input = first 99 chars, target = the same shifted by one

X_train, y_train = make_sequences(text_as_int[:train_end])
X_val,   y_val   = make_sequences(text_as_int[train_end:val_end])
X_test,  y_test  = make_sequences(text_as_int[val_end:])

print(f"Training sequences  : {len(X_train):,}   (from {train_end:,} characters)")
print(f"Validation sequences: {len(X_val):,}   (from {val_end - train_end:,} characters)")
print(f"Test sequences      : {len(X_test):,}   (from {n - val_end:,} characters)")
print("X shape:", X_train.shape, "  y shape:", y_train.shape)

## 4. Build and train the three models

In [ ]:
EPOCHS = 10
BATCH_SIZE = 128

def build_model(recurrent_layer, name):
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=64),
        recurrent_layer(128, return_sequences=True),
        Dense(vocab_size, activation="softmax"),
    ], name=name)
    model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["sparse_categorical_accuracy"])
    return model

architectures = {"Simple RNN": SimpleRNN, "LSTM": LSTM, "GRU": GRU}
models, histories = {}, {}

for name, layer in architectures.items():
    print("=" * 70)
    print("Training", name)
    print("=" * 70)
    model = build_model(layer, name.replace(" ", "_").lower())
    model.build(input_shape=(None, sequence_length - 1))
    model.summary()

    stopper = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
    history = model.fit(X_train, y_train,
                        validation_data=(X_val, y_val),
                        batch_size=BATCH_SIZE, epochs=EPOCHS,
                        callbacks=[stopper], verbose=2)
    models[name], histories[name] = model, history

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, (name, history) in zip(axes, histories.items()):
    ax.plot(history.history["loss"], label="Training loss")
    ax.plot(history.history["val_loss"], label="Validation loss")
    ax.set_title(name)
    ax.set_xlabel("Epoch")
    ax.legend()
axes[0].set_ylabel("Loss")
plt.suptitle("Training and validation loss")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
for name, history in histories.items():
    plt.plot(history.history["loss"], label=name)
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("RNN vs LSTM vs GRU - training loss")
plt.legend()
plt.show()

## 5. Generate text

In [ ]:
GEN_LEN = 500
context_length = sequence_length - 1

def generate_text(model, seed_text, num_generate=300, temperature=0.7):
    seed_text = seed_text.lower()
    generated_text = seed_text

    for _ in range(num_generate):
        # last characters as integers (unknown characters -> 0)
        input_sequence = [char_to_index.get(c, 0) for c in generated_text[-context_length:]]

        # pad on the left if the text is still short
        if len(input_sequence) < context_length:
            input_sequence = [0] * (context_length - len(input_sequence)) + input_sequence

        predictions = model(np.array([input_sequence]), training=False).numpy()[0, -1, :]

        # temperature controls randomness: low = safe and repetitive, high = creative and messy
        predictions = np.log(predictions + 1e-8) / temperature
        probabilities = tf.nn.softmax(predictions).numpy()

        next_index = np.random.choice(len(probabilities), p=probabilities)
        generated_text += index_to_char[next_index]

    return generated_text

seed = "to be or not to be"
for name, model in models.items():
    print("=" * 70)
    print(name, " (temperature 0.7)")
    print("=" * 70)
    print(generate_text(model, seed, num_generate=GEN_LEN, temperature=0.7))
    print()

In [ ]:
seed = "love is"
for name, model in models.items():
    print("=" * 70)
    print(name, "- seed:", repr(seed))
    print("=" * 70)
    print(generate_text(model, seed, num_generate=300, temperature=0.7))
    print()

---
# Testing the language models

1. **Held-out test text** - loss, perplexity and next-character accuracy on text the models never saw. A *unigram*
   baseline (always guess by character frequency) shows whether the models learned anything beyond letter counts.
2. **Next-character prediction examples** with top-1 and top-3 accuracy.
3. **Quality of generated text** - how much of the output consists of real Shakespeare words, at three temperatures.

## Test 1: loss, perplexity and accuracy on the test text

In [ ]:
# Unigram baseline: predict every character from its frequency in the training text
counts = np.bincount(text_as_int[:train_end], minlength=vocab_size) + 1
unigram_p = counts / counts.sum()
unigram_loss = -np.mean(np.log(unigram_p[y_test]))
unigram_acc = float((y_test == unigram_p.argmax()).mean())

rows = [{
    "Model": "Unigram baseline (letter frequency)", "Parameters": 0, "Epochs trained": 0,
    "Train loss": "-", "Val loss": "-",
    "Test loss": round(float(unigram_loss), 3), "Test perplexity": round(float(np.exp(unigram_loss)), 2),
    "Test char accuracy": f"{unigram_acc * 100:.1f}%",
}]

for name, model in models.items():
    h = histories[name].history
    best = int(np.argmin(h["val_loss"]))
    test_loss, test_acc = model.evaluate(X_test, y_test, batch_size=256, verbose=0)
    rows.append({
        "Model": name, "Parameters": model.count_params(), "Epochs trained": len(h["loss"]),
        "Train loss": round(h["loss"][best], 3), "Val loss": round(h["val_loss"][best], 3),
        "Test loss": round(test_loss, 3), "Test perplexity": round(float(np.exp(test_loss)), 2),
        "Test char accuracy": f"{test_acc * 100:.1f}%",
    })

results_df = pd.DataFrame(rows).set_index("Model")
display(results_df)

best_model_name = results_df.drop(index="Unigram baseline (letter frequency)")["Test loss"].astype(float).idxmin()
print("Best model on the test text:", best_model_name)
print("Perplexity = how many characters the model is 'choosing between' on average; lower is better.")

plot_df = results_df.drop(index="Unigram baseline (letter frequency)")
plt.figure(figsize=(7, 4))
plt.bar(plot_df.index, plot_df["Test perplexity"].astype(float))
plt.axhline(float(np.exp(unigram_loss)), color="red", linestyle="--", label="Unigram baseline")
plt.ylabel("Test perplexity (lower is better)")
plt.title("Perplexity on unseen text")
plt.legend()
plt.show()

## Test 2: next-character prediction

In [ ]:
rng = np.random.default_rng(0)
subset = rng.choice(len(X_test), size=min(2000, len(X_test)), replace=False)
Xs, ys = X_test[subset], y_test[subset][:, -1]          # predict the character after each 99-character context

rows, all_probs = [], {}
for name, model in models.items():
    probs = model.predict(Xs, batch_size=256, verbose=0)[:, -1, :]
    all_probs[name] = probs
    top3 = np.argsort(-probs, axis=1)[:, :3]
    rows.append({
        "Model": name,
        "Top-1 accuracy": f"{(top3[:, 0] == ys).mean() * 100:.1f}%",
        "Top-3 accuracy": f"{np.mean([ys[i] in top3[i] for i in range(len(ys))]) * 100:.1f}%",
    })
display(pd.DataFrame(rows).set_index("Model"))

# a few readable examples from the best model
probs = all_probs[best_model_name]
rows = []
for i in range(6):
    context = "".join(index_to_char[Xs[i][-30:]]).replace("\n", "\\n")
    top3 = np.argsort(-probs[i])[:3]
    rows.append({
        "Last 30 characters of the context": context,
        "Actual next": repr(str(index_to_char[ys[i]])),
        "Top-3 predictions": ", ".join(f"{str(index_to_char[t])!r} ({probs[i][t] * 100:.0f}%)" for t in top3),
    })
print(f"\nExamples from {best_model_name}:")
display(pd.DataFrame(rows))

## Test 3: how good is the generated text?

In [ ]:
# Real words = words that occur in the training text
real_words = set(re.findall(r"[a-z']+", text[:train_end]))

def real_word_share(generated):
    words = re.findall(r"[a-z']+", generated)
    return (sum(w in real_words for w in words) / len(words) * 100) if words else 0.0

temperatures = [0.3, 0.7, 1.0]
rows, samples = [], {}
for name, model in models.items():
    for t in temperatures:
        generated = generate_text(model, "to be or not to be", num_generate=GEN_LEN, temperature=t)
        samples[(name, t)] = generated
        rows.append({"Model": name, "Temperature": t,
                     "Real words": f"{real_word_share(generated):.1f}%",
                     "Distinct characters used": len(set(generated))})

quality_df = pd.DataFrame(rows)
display(quality_df)

pivot = quality_df.assign(share=quality_df["Real words"].str.rstrip("%").astype(float)).pivot(
    index="Temperature", columns="Model", values="share")
pivot.plot(marker="o", figsize=(7, 4))
plt.ylabel("Real words in generated text (%)")
plt.title("Quality of generated text vs. temperature")
plt.show()

print("Sample from the best model at the lowest temperature:\n")
print(samples[(best_model_name, 0.3)][:400])

## Conclusion

Three character-level models (Simple RNN, LSTM, GRU) were trained on Shakespeare text and used to generate new
text. Because the text was split by position, the testing section could measure loss, perplexity and next-character
accuracy on a genuinely unseen part of the play and compare them with a letter-frequency baseline. The generated
text was scored by the share of real words it contains at different temperatures: low temperatures give safer,
more repetitive text, high temperatures give more varied but less correct text.